In [1]:
import os

In [10]:
from  langchain_groq import ChatGroq 

In [11]:
llm_model = ChatGroq(model="qwen/qwen3-32b")

In [7]:
llm_model.invoke("weather in coimbatore, india tody ?")

AIMessage(content="<think>\nOkay, the user is asking for the weather in Coimbatore, India today. Let me start by checking if I have the current data. Wait, I remember that my knowledge cutoff is October 2023, so I can't provide real-time weather updates. I should inform them about that first.\n\nNext, I need to explain that I can't access live weather info but maybe offer some general advice. Maybe suggest checking a reliable weather app or website. Also, consider the user's possible needs. Are they planning an outdoor activity? Maybe mention typical weather patterns for Coimbatore during this time of year if I know. Let me recall, Coimbatore is in Tamil Nadu, a tropical region. Depending on the season, the weather could be hot and dry or maybe it's monsoon season. Wait, as of October, the monsoon usually retreats by September, so October might be transitioning into winter, but I should be careful not to assume. Maybe just mention that temperatures can vary and suggest checking a forec

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [13]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [10]:
embedding_model.embed_query("hello world")

[-0.030033651739358902,
 0.004021478816866875,
 0.01552528329193592,
 -0.08638038486242294,
 -0.0035104958806186914,
 0.014537660405039787,
 -0.013522510416805744,
 -0.001957012340426445,
 0.01483400072902441,
 -0.003284806152805686,
 -0.011062352918088436,
 -0.010215921327471733,
 0.007057697977870703,
 0.012086395174264908,
 0.16365478932857513,
 -0.002110831206664443,
 -0.0027889995835721493,
 -0.0039434246718883514,
 -8.16034444142133e-05,
 -0.02011292800307274,
 0.00675996346399188,
 0.006364765577018261,
 0.013239339925348759,
 -0.004769574850797653,
 -0.016854675486683846,
 0.005877794232219458,
 0.019046057015657425,
 0.0019360381411388516,
 0.013154467567801476,
 -0.004181211814284325,
 0.0009924883488565683,
 0.012267115525901318,
 -0.007287244312465191,
 0.013363276608288288,
 -0.00428686011582613,
 0.016898537054657936,
 0.019980784505605698,
 0.008453992195427418,
 -0.0014555914094671607,
 -0.0005566618638113141,
 0.00048513340880163014,
 0.0025110847782343626,
 0.00455104

In [14]:
from langchain_community.document_loaders import PyPDFLoader

In [15]:
file_path = os.path.join(os.getcwd(),"data","Dashboard Summary.pdf")
file_path

'd:\\MyData\\Learnings\\AI\\LLMOps\\document_portal\\github\\document-portal\\notebook\\data\\Dashboard Summary.pdf'

In [16]:
doc_load = PyPDFLoader(file_path=file_path)

In [17]:
documents = doc_load.load()

In [18]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [19]:
text_Splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,length_function=len)

In [20]:
docs = text_Splitter.split_documents(documents)

In [21]:
from langchain_community.vectorstores import FAISS

In [22]:
vector_store = FAISS.from_documents(documents=docs, embedding=embedding_model)

In [23]:
retriever = vector_store.as_retriever()

In [24]:
vector_store.similarity_search("how many percentage of deals closed in tier 2 clients ?")

[Document(id='99720cfe-d363-4930-b15f-b2ae6f1b7c35', metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-07-24T17:53:05+05:30', 'author': 'Admin', 'moddate': '2025-07-24T17:53:05+05:30', 'title': 'Microsoft Word - Dashboard Summary', 'source': 'd:\\MyData\\Learnings\\AI\\LLMOps\\document_portal\\github\\document-portal\\notebook\\data\\Dashboard Summary.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Project Code: XR-2241 — Quarterly Pipeline Review \nHighlights: \n• 72% of deals closed were from Tier 2 clients \n• Average sales cycle decreased to 18 days \n• Top performer: East Region with ₹1.2Cr in revenue \n• AI Lead Scoring contributed to a 15% uplift in conversion rates \n \nAction Items: \n- Retrain the model on updated client signals \n- Expand region-specific playbooks')]

In [25]:
prompt_template ="""
You are my document analyser.  You can review the document from the given context and answer the user question.
If you don't have enough information then reply user that 'I don't have enougn information'

context ===>  {context}

question ===> {question}

answer   ===>
"""

In [26]:
i:int=0
for doc in docs:
    print(f"doc {i} : \n {doc}")


doc 0 : 
 page_content='Project Code: XR-2241 — Quarterly Pipeline Review 
Highlights: 
• 72% of deals closed were from Tier 2 clients 
• Average sales cycle decreased to 18 days 
• Top performer: East Region with ₹1.2Cr in revenue 
• AI Lead Scoring contributed to a 15% uplift in conversion rates 
 
Action Items: 
- Retrain the model on updated client signals 
- Expand region-specific playbooks' metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-07-24T17:53:05+05:30', 'author': 'Admin', 'moddate': '2025-07-24T17:53:05+05:30', 'title': 'Microsoft Word - Dashboard Summary', 'source': 'd:\\MyData\\Learnings\\AI\\LLMOps\\document_portal\\github\\document-portal\\notebook\\data\\Dashboard Summary.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [26]:
def format_doc(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [27]:
from langchain.prompts import PromptTemplate

In [28]:
prompt = PromptTemplate(
    template=prompt_template ,
    input_variables=["context","question"]
)

In [29]:
from langchain_core.runnables import  RunnablePassthrough

In [30]:
chain ={"context":retriever | format_doc, "question":RunnablePassthrough()}| prompt | llm_model 

In [31]:
chain.invoke("what is the decreased sales ?")

AIMessage(content='<think>\nOkay, let\'s see. The user is asking "what is the decreased sales?" referring to the context provided.\n\nFirst, I need to look at the context details. The context mentions "Average sales cycle decreased to 18 days." The term "decreased sales" might be a bit ambiguous, but in this context, it\'s likely referring to the reduction in the average sales cycle. The user might be asking how much the sales cycle has been shortened. \n\nIn the highlights, it\'s stated that the average sales cycle is now 18 days. However, there\'s no previous figure provided to compare it to. Without knowing the previous duration, I can\'t specify the exact decrease in days. The context doesn\'t mention the prior average sales cycle length. \n\nSince the user is asking for the "decreased sales," which could be a typo for "decreased sales cycle," but even then, without the original number, I can\'t calculate the decrease. The answer should point out that the average sales cycle is now

# Document Compare

In [32]:
document_compare_prompt_template="""
You will be provided 2 versions of same PDF.  You tasks are as follows:

1. Identify the difference between versions
2. Make a note of pdf page and line number
3. Print the difference along with in which page and line number you find the difference

Input  document :

{combined_documents}

Your response should follow this format

{format_instruction}

"""

In [79]:
compare_prompt = PromptTemplate(
    template=document_compare_prompt_template,
    input_variables=["combined_documents","format_instruction"]
)

In [3]:
from pydantic import BaseModel, RootModel

In [4]:
class ChangeFormat(BaseModel):
    pageno: int
    lineno: int
    changes:str

In [5]:
class SummaryResponse(RootModel[list[ChangeFormat]]):
    pass

In [1]:
from langchain_core.output_parsers import JsonOutputParser


In [6]:
parser = JsonOutputParser(pydantic_object=SummaryResponse)


In [82]:
from langchain.output_parsers import OutputFixingParser


In [83]:
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm_model)


In [84]:
chain = compare_prompt | llm_model | fixing_parser 

In [39]:
from  pathlib import Path

In [43]:
doc_path = Path(os.getcwd()) / "data/compare"

In [52]:
import  fitz


In [68]:
def readpdf(filename):
    with fitz.open(filename) as doc:
        if doc.is_encrypted:
            raise(f"file {filename} is encrypted")
        all_text=[]
        for pagenum in range(doc.page_count):
            page = doc.load_page(pagenum)
            text = page.get_text()
            if text.strip():
                all_text.append(f"\n -- page {pagenum} -- \n {text}")
        return "\n".join(all_text)

In [75]:
content_dict ={}
for filename in sorted(doc_path.iterdir()):
    if filename.is_file() and filename.suffix==".pdf":
        content_dict[filename.name] = readpdf(filename)

combined_documents=[]
for filename,content in content_dict.items():
    combined_documents.append(f"Documents: {filename} \n {content}")
    

        


In [77]:
inputs = {
    "combined_documents" : combined_documents,
    "format_instruction" : parser.get_format_instructions()
}

In [85]:
response =chain.invoke(inputs)

In [86]:
print(response)

[{'pageno': 0, 'lineno': 4, 'changes': "STATUS changed from 'NORMAL' to 'OFF SPEC'"}, {'pageno': 0, 'lineno': 49, 'changes': "Test Result changed from '0.1' to '0.7'"}]


In [87]:
import pandas as pd


In [88]:
df=pd.DataFrame(response)

In [89]:
df

,pageno,lineno,changes
0,0,4,STATUS changed from 'NORMAL' to 'OFF SPEC'
1,0,49,Test Result changed from '0.1' to '0.7'
